In [2]:
# Forzamos la versión estable de NumPy y actualizamos pyarrow
%pip install "numpy<2.0.0" "pyarrow>=14.0.1" "pandas>=2.0.0" --force-reinstall

  Using cached pyarrow-23.0.1-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   ---------------------------------------  15.5/15.5 MB 107.8 MB/s eta 0:00:01
   ---------------------------------------- 15.5/15.5 MB 75.2 MB/s eta 0:00:00
Using cached pyarrow-23.0.1-cp312-cp312-win_amd64.whl (27.6 MB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 50.7 MB/s eta 0:00:00
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.2
    Uninstalling tzdata-2025.2:
      Successfully uninstalled tzdata-2025.2
  Attempting uninstall: six
    Found existing installation: 

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
import os
import subprocess
import sys

# 1. FORZAR DOWNGRADE DE NUMPY (Si sigue saliendo 2.1.3)
try:
    import numpy as np
    if np.__version__.startswith('2'):
        print("🔄 Bajando versión de NumPy para compatibilidad...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy<2.0.0"])
        print("✅ NumPy bajado. REINICIA EL KERNEL AHORA y no corras esta celda otra vez.")
except:
    pass

import pandas as pd

# 2. CONFIGURAR RUTA Y BUSCAR EL ARCHIVO REAL
folder_path = r'C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados'

if os.path.exists(folder_path):
    files = os.listdir(folder_path)
    print(f"Archivos encontrados en la carpeta: {files}")
    
    # Buscamos cualquier archivo que termine en .parquet para no fallar por el nombre
    parquet_files = [f for f in files if f.endswith('.parquet')]
    
    if parquet_files:
        real_path = os.path.join(folder_path, parquet_files[0])
        print(f"📂 Intentando leer: {real_path}")
        
        try:
            # Intentamos leer con 'fastparquet' que a veces es más tolerante con NumPy 2.0
            # Si no tienes fastparquet, corre: %pip install fastparquet
            df = pd.read_parquet(real_path, engine='pyarrow')
            print("\n🚀 ¡ÉXITO! Datos cargados.")
            print(f"Filas: {len(df):,}")
            print(df['Label'].value_counts())
        except Exception as e:
            print(f"❌ Error al leer el archivo: {e}")
    else:
        print("❌ No hay archivos .parquet en esa carpeta. ¿Seguro que el script de unificación terminó bien?")
else:
    print("❌ La ruta de la carpeta no existe. Revisa si 'Datasets unificados' tiene espacios extra.")

Archivos encontrados en la carpeta: ['CICIDS2017', 'CICIDS2017.md', 'CICIDS2017_unido.parquet', 'CICIDS2017_Unificacion.py', 'Darknet.csv', 'UNSW-NB15-V3.csv', 'Verificación-CICIDS.ipynb']
📂 Intentando leer: C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados\CICIDS2017_unido.parquet

🚀 ¡ÉXITO! Datos cargados.
Filas: 2,830,743
Label
BENIGN                          2273097
DoS Hulk                         231073
PortScan                         158930
DDoS                             128027
DoS GoldenEye                     10293
FTP-Patator                        7938
SSH-Patator                        5897
DoS slowloris                      5796
DoS Slowhttptest                   5499
Bot                                1966
Web Attack ï¿½ Brute Force         1507
Web Attack ï¿½ XSS                  652
Infiltration                         36
Web Attack ï¿½ Sql Injection         21
Heartbleed                           11
Name: count, dtype: int64


In [3]:
print(f"Número de columnas: {df.shape[1]}")
print("\nListado de características:")
print(df.columns)

Número de columnas: 79

Listado de características:
Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Len

In [4]:
import pandas as pd
import numpy as np

# 1. Limpiar los nombres de los ataques (Eliminar caracteres raros)
# Reemplazamos el caracter raro por un guion para que sea legible
df['Label'] = df['Label'].str.replace('ï¿½', '-', regex=False)

print("✅ Labels normalizados:")
print(df['Label'].unique())

# 2. Identificar columnas numéricas
cols_num = df.select_dtypes(include=[np.number]).columns

# 3. Manejo de Infinitos (Estrategia: Reemplazar por el Máximo de la columna)
# Primero pasamos los inf a NaN temporalmente para calcular el máximo real
df[cols_num] = df[cols_num].replace([np.inf, -np.inf], np.nan)

# Llenamos los NaN (que antes eran inf o nulos) con el valor máximo de cada columna
# Esto es mejor que poner 0, porque el 'inf' suele indicar una tasa de tráfico altísima
df[cols_num] = df[cols_num].fillna(df[cols_num].max())

# 4. Verificación final
inf_final = np.isinf(df[cols_num]).values.sum()
nan_final = df.isnull().sum().sum()

print(f"\n--- Verificación de Limpieza ---")
print(f"Valores Infinitos restantes: {inf_final}")
print(f"Valores Nulos (NaN) restantes: {nan_final}")

✅ Labels normalizados:
<ArrowStringArray>
[                    'BENIGN',                       'DDoS',
                   'PortScan',                        'Bot',
               'Infiltration',   'Web Attack - Brute Force',
           'Web Attack - XSS', 'Web Attack - Sql Injection',
                'FTP-Patator',                'SSH-Patator',
              'DoS slowloris',           'DoS Slowhttptest',
                   'DoS Hulk',              'DoS GoldenEye',
                 'Heartbleed']
Length: 15, dtype: str

--- Verificación de Limpieza ---
Valores Infinitos restantes: 0
Valores Nulos (NaN) restantes: 0


In [5]:
output_clean = r'C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados\CICIDS2017.parquet'
df.to_parquet(output_clean, index=False)
print(f"💾 Dataset limpio guardado en: {output_clean}")

💾 Dataset limpio guardado en: C:\Users\Felix\Desktop\Tesis\Datasets\Datasets unificados\CICIDS2017.parquet
